In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Full ETL Pipeline Using RDD → DataFrame → SQL
```
We’ll simulate a small log file ETL pipeline:

Extract: read raw text logs into an RDD

Transform: parse logs, filter errors, extract fields

Load: convert to DataFrame

Analyze: run SQL queries

Summarize: produce final aggregated results
```

# STEP 1 — Setup Spark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("FullRDD_ETL").getOrCreate()
sc = spark.sparkContext


# STEP 2 — Create a Raw Log File (Simulated)
- Instead of reading a real file, we create a list of log lines.

In [3]:
raw_logs = [
    "2024-01-01 INFO Application started",
    "2024-01-01 WARN Low memory",
    "2024-01-01 ERROR Disk failure",
    "2024-01-02 INFO User login",
    "2024-01-02 ERROR Network timeout",
    "2024-01-03 INFO Shutdown complete"
]

rdd_logs = sc.parallelize(raw_logs)

# STEP 3 — Parse Logs Using RDD Transformations
```
We extract:

date

level (INFO/WARN/ERROR)

message
```

In [10]:
rdd_parsed = (
    rdd_logs.map(lambda line: line.split(" ", 2))
            .map(lambda parts: (parts[0], parts[1], parts[2]))
)

In [11]:
print(rdd_parsed.collect())

[('2024-01-01', 'INFO', 'Application started'), ('2024-01-01', 'WARN', 'Low memory'), ('2024-01-01', 'ERROR', 'Disk failure'), ('2024-01-02', 'INFO', 'User login'), ('2024-01-02', 'ERROR', 'Network timeout'), ('2024-01-03', 'INFO', 'Shutdown complete')]


# STEP 4 — Convert RDD → DataFrame

In [15]:
df_logs = rdd_parsed.toDF(["date", "level", "message"])
df_logs.show()


+----------+-----+-------------------+
|      date|level|            message|
+----------+-----+-------------------+
|2024-01-01| INFO|Application started|
|2024-01-01| WARN|         Low memory|
|2024-01-01|ERROR|       Disk failure|
|2024-01-02| INFO|         User login|
|2024-01-02|ERROR|    Network timeout|
|2024-01-03| INFO|  Shutdown complete|
+----------+-----+-------------------+



# STEP 5 — Register as SQL Table

In [18]:
df_logs.createOrReplaceTempView("logs_vw")


# STEP 6 — SQL Analytics

In [24]:
## 1. Count logs by level
spark.sql("""
    SELECT level, COUNT(*) AS count
    FROM logs
    GROUP BY level
    ORDER BY count DESC
""").show()


+-----+-----+
|level|count|
+-----+-----+
| INFO|    3|
|ERROR|    2|
| WARN|    1|
+-----+-----+



In [25]:
## 2. Extract only ERROR logs
spark.sql("""
    SELECT date, message
    FROM logs
    WHERE level = 'ERROR'
""").show()


+----------+---------------+
|      date|        message|
+----------+---------------+
|2024-01-01|   Disk failure|
|2024-01-02|Network timeout|
+----------+---------------+



In [26]:
## 3. Count logs per day
spark.sql("""
    SELECT date, COUNT(*) AS total_logs
    FROM logs
    GROUP BY date
    ORDER BY date
""").show()


+----------+----------+
|      date|total_logs|
+----------+----------+
|2024-01-01|         3|
|2024-01-02|         2|
|2024-01-03|         1|
+----------+----------+



# STEP 7 — Final Summary DataFrame

In [27]:
df_summary = spark.sql("""
    SELECT 
        level,
        COUNT(*) AS total,
        MIN(date) AS first_seen,
        MAX(date) AS last_seen
    FROM logs
    GROUP BY level
""")

df_summary.show()


+-----+-----+----------+----------+
|level|total|first_seen| last_seen|
+-----+-----+----------+----------+
|ERROR|    2|2024-01-01|2024-01-02|
| INFO|    3|2024-01-01|2024-01-03|
| WARN|    1|2024-01-01|2024-01-01|
+-----+-----+----------+----------+

